## Setup and Training Word2Vec

In [ ]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

df = pd.read_csv('../data/processed/cleaned_spam.csv').dropna()
X_train, X_test, y_train, y_test = train_test_split(df['clean_message'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])

# Word2Vec requires lists of tokens
sentences_train = [text.split() for text in X_train]
sentences_test = [text.split() for text in X_test]

# Train Word2Vec on our specific corpus
w2v_model = Word2Vec(sentences=sentences_train, vector_size=100, window=5, min_count=1, workers=4)

## Averaging Vectors and Training

In [ ]:
# Function to average word vectors for a whole message
def get_sentence_vector(tokens, model, vector_size):
    valid_tokens = [model.wv[word] for word in tokens if word in model.wv]
    if not valid_tokens:
        return np.zeros(vector_size)
    return np.mean(valid_tokens, axis=0)

X_train_w2v = np.array([get_sentence_vector(tokens, w2v_model, 100) for tokens in sentences_train])
X_test_w2v = np.array([get_sentence_vector(tokens, w2v_model, 100) for tokens in sentences_test])

# Train classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_w2v, y_train)

print("Word2Vec + Logistic Regression:")
print(classification_report(y_test, clf.predict(X_test_w2v)))